# Comparaison des méthodes de classification supervisée — Régression logistique pénalisée
**Données** : `SAh.csv` — cohorte sud-africaine, cible = `chd`  
**Référence** : https://regression-avec-python.github.io/codes/chap15.html  
**Plan** :
1. **Variables de base** : Logistique non-pénalisée, Lasso, ElasticNet, Ridge (CV 10 blocs)
2. **Feature engineering avec interactions** : termes d'ordre 2 entre toutes les variables
3. **Feature engineering polynomial** : termes quadratiques et cubiques  
→ Pour chaque jeu de variables, on compare les méthodes par taux d'erreur (seuil 0.5 et seuil naturel)

## Note sur la pénalisation dans sklearn
En régression logistique sklearn, la **pénalisation est active par défaut** (L2, `C=1`).  
Pour les méthodes Lasso/Ridge/ElasticNet, il faut définir la grille de C (= 1/λ) via la fonction `grille()` ci-dessous, car `LogisticRegressionCV` ne génère pas automatiquement une grille aussi bien calibrée que `LassoCV` en régression.

In [1]:
import pandas as pd; import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, \
    LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
import sklearn.metrics as sklm
from patsy import dmatrix

import sys
sys.path.append('../modules')
import logistic_step_sk as lss 

In [ ]:
# Chargement et préparation des données (normalement fait dans preparation.ipynb)
don = pd.read_csv("https://regression-avec-python.github.io/donnees/SAh.csv", header=0, sep=",")
don.rename(columns={"chd": "Y"}, inplace=True)
don.Y.value_counts()    # 302 chd=0, 160 chd=1 → proportion ~34% positifs

# Séparation variables quantitatives / qualitatives
X = don.drop(columns=["Y"])
Y = don.filter(["Y"])
Xquanti = X.select_dtypes(exclude=['object'])
Xquali  = X.select_dtypes(include=['object'])

# Encodage des variables qualitatives (famhist : Present/Absent → 1/0)
XqualiD = pd.get_dummies(Xquali, drop_first=True, dtype=float)
Xbase = pd.concat([Xquanti, XqualiD], axis=1)  # X de base : variables originales + encodage

   sbp  tobacco   ldl  adiposity  typea  obesity  alcohol  age
0  160    12.00  5.73      23.11     49    25.30    97.20   52
1  144     0.01  4.41      28.61     55    28.87     2.06   63
2  118     0.08  3.48      32.28     52    29.14     3.81   46
   famhist
0  Present
1   Absent
2  Present


In [ ]:
# Y en numpy + tableau de résultats PROB (une colonne par méthode)
Y = don["Y"].to_numpy()
PROB = pd.DataFrame({"Y": Y, "log": 0.0, "BIC": 0.0, "AIC": 0.0,
                     "ridge": 0.0, "lasso": 0.0, "elast": 0.0})

In [ ]:
# Construction de X via dmatrix (encode famhist, ajoute interactions si nécessaire)
nomsvar = don.columns.difference(["Y"])
formule = "~" + "+".join(nomsvar)
X = dmatrix(formule, don, return_type="dataframe").iloc[:, 1:].to_numpy()  # supprime intercept

In [ ]:
nb = 10
# StratifiedKFold : préserve la proportion de Y=1 dans chaque fold (important car ~34% positifs)
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=123)

In [59]:
# on crée les grilles des lambda
def grille(X, y, type = "lasso", ng=400):
    scalerX = StandardScaler().fit(X)
    Xcr= scalerX.transform(X)
    l0 = np.abs(Xcr.transpose().dot((y-y.mean()))).max()/X.shape[0]
    llc = np.linspace(0,-4,ng)
    ll = l0*10**llc
    if type=="lasso":
        Cs = 1/ 0.9/ X.shape[0] / (l0*10**(llc))
    elif type=="ridge":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 100)
    elif type=="enet":
        Cs = 1/ 0.9/ X.shape[0] / ((l0*10**(llc)) * 2)
    return Cs

In [60]:
# code modifié par rapport au livre
for app_index, val_index in skf.split(X,Y):
    Xapp = X[app_index,:]
    Xtest = X[val_index,:]
    Yapp = Y[app_index]
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    #choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="bic",m).fit(Xapp,Yapp)
    #PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    #choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[],direction="forward",crit="aic").fit(Xapp,Yapp)
    #PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2",Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]

In [61]:
round(PROB.iloc[0:4,:],3)

,Y,log,BIC,AIC,ridge,lasso,elast
0,1,0.742,0.0,0.0,0.675,0.587,0.598
1,1,0.292,0.0,0.0,0.321,0.373,0.358
2,0,0.251,0.0,0.0,0.286,0.316,0.297
3,1,0.719,0.0,0.0,0.676,0.694,0.682


In [62]:
# création d'un fichier contenant toutes les probas prévues après CV
PROB.to_csv(("PROB.csv"), index=False)

In [63]:
PROB

,Y,log,BIC,AIC,ridge,lasso,elast
0,1,0.741916,0.0,0.0,0.675463,0.587063,0.597780
1,1,0.292273,0.0,0.0,0.320946,0.373265,0.358088
2,0,0.250513,0.0,0.0,0.286156,0.316315,0.296510
3,1,0.718669,0.0,0.0,0.676079,0.694443,0.681593
4,1,0.712518,0.0,0.0,0.641409,0.517724,0.537167
...,...,...,...,...,...,...,...
457,0,0.675269,0.0,0.0,0.597518,0.573582,0.563190
458,1,0.325507,0.0,0.0,0.357781,0.325385,0.333119
459,0,0.125418,0.0,0.0,0.150607,0.186509,0.187677
460,0,0.541794,0.0,0.0,0.513922,0.470703,0.521638


# Analyse de la Classification supervisée

In [38]:
# Comparaison des méthodes en fixant s = 0,5
mc = pd.Series(0.0, index=PROB.columns[1:])
s = 0.5
for i in range(mc.shape[0]):
    mc.iloc[i] = sklm.zero_one_loss(PROB.Y, PROB.iloc[:,i+1]>s)
round(mc,3)

log      0.281
BIC      0.346
AIC      0.346
ridge    0.266
lasso    0.279
elast    0.271
dtype: float64

Calcul des indicateurs selon le seuil s

In [43]:
import sklearn.metrics as sklm
noms = PROB.columns[1:]
matsB = pd.DataFrame({"seuil": pd.Series(0.0, index=noms)})
s = .5
for nom in noms:
    matsB.loc[nom,"seuil"] = s
    matsB.loc[nom,"accuracy"] = sklm.accuracy_score(PROB.Y, PROB.loc[:,nom]>=s)
    confmat = sklm.confusion_matrix(PROB.Y, PROB.loc[:,nom]>=s)
    matsB.loc[nom,"sensitivity"] = confmat[1,1]/(confmat[1,1]+confmat[1,0])
    matsB.loc[nom,"specificity"] = confmat[0,0]/(confmat[0,0]+confmat[0,1])
    matsB.loc[nom,"medecin"] = matsB.loc[nom,"sensitivity"]+matsB.loc[nom,"specificity"]
    matsB.loc[nom,"F1"] = sklm.f1_score(PROB.Y, PROB.loc[:,nom]>=s)
print(matsB.round(3))

        seuil  accuracy  sensitivity  specificity  medecin     F1
log       0.5     0.719        0.512        0.828    1.340  0.558
BIC       0.5     0.654        0.000        1.000    1.000  0.000
AIC       0.5     0.654        0.000        1.000    1.000  0.000
ridge     0.5     0.734        0.481        0.868    1.349  0.556
lasso     0.5     0.721        0.400        0.891    1.291  0.498
elast     0.5     0.729        0.431        0.887    1.319  0.525
LassoL    0.5     0.721        0.481        0.848    1.329  0.544
RidgeL    0.5     0.723        0.481        0.851    1.332  0.546
EnetL     0.5     0.727        0.475        0.861    1.336  0.547
LassoA    0.5     0.732        0.456        0.877    1.334  0.541
RidgeA    0.5     0.716        0.375        0.897    1.272  0.478
EnetA     0.5     0.729        0.425        0.891    1.316  0.521


Calcul des indicateurs en déterminant s selon le s naturel (= proportion de 1 dans les Y initial pour respecter les proportions de l'échantillon initial

In [51]:
matsN = pd.DataFrame({"seuil": pd.Series(0.0, index=noms)})
noms = PROB.columns[1:]
nbr0 = PROB.Y.value_counts()[0]
for nom in noms:
    tmp = PROB.loc[:,nom].sort_values(ascending=True)
    s = (tmp.iloc[nbr0-1]+tmp.iloc[nbr0])/2
    matsN.loc[nom,"seuil"] = s
    matsN.loc[nom,"accuracy"] = sklm.accuracy_score(PROB.Y, PROB.loc[:,nom]>=s)
    confmat = sklm.confusion_matrix(PROB.Y, PROB.loc[:,nom]>=s)
  #  matsN.loc[nom, "tn"] = confmat[0,0]
  #  matsN.loc[nom, "tp"] = confmat[1,1]
  #  matsN.loc[nom, "fn"] = confmat[1,0]
  #  matsN.loc[nom, "fp"] = confmat[0,1]
    matsN.loc[nom,"sensitivity"] = confmat[1,1]/(confmat[1,1]+confmat[1,0])
    matsN.loc[nom,"specificity"] = confmat[0,0]/(confmat[0,0]+confmat[0,1])
    matsN.loc[nom,"medecin"] = matsN.loc[nom,"sensitivity"]+matsN.loc[nom,"specificity"]
    matsN.loc[nom,"F1"] = sklm.f1_score(PROB.Y, PROB.loc[:,nom]>=s)
print(matsN.round(3))

        seuil  accuracy  sensitivity  specificity  medecin     F1
log     0.431     0.714        0.588        0.781    1.369  0.588
BIC     0.000     0.346        1.000        0.000    1.000  0.514
AIC     0.000     0.346        1.000        0.000    1.000  0.514
ridge   0.418     0.719        0.594        0.785    1.379  0.594
lasso   0.429     0.714        0.588        0.781    1.369  0.588
elast   0.425     0.723        0.600        0.788    1.388  0.600
LassoL  0.442     0.710        0.581        0.778    1.359  0.581
RidgeL  0.433     0.706        0.575        0.775    1.350  0.575
EnetL   0.437     0.710        0.581        0.778    1.359  0.581
LassoA  0.431     0.714        0.588        0.781    1.369  0.588
RidgeA  0.418     0.714        0.588        0.781    1.369  0.588
EnetA   0.425     0.701        0.569        0.772    1.340  0.569


Conclusion : Elasticnet gagne pour le medecin (optimisation selon sensibilité et spécificité) mais également sur les autres critères

### Etape suivante : Feature engineering pour optimiser prédiction

Car on a très peu de variables / au nombre de lignes => augmenter le nombre de variables (= l'information)
Idée du ratio nb paramètres p par rapport au nombre d'individus n : 50 valeurs(individus) / paramètre

=> après feature engineering on relance les modèles et on espère optimiser les critères

## Partie 2 — Feature engineering : interactions entre variables
On ajoute les termes d'ordre 2 (produits croisés entre toutes les variables) via la formule `patsy` `(X1+X2+...)**2`.  
Cela augmente le nombre de variables de p à p(p+1)/2, permettant au modèle de capturer des effets non-linéaires.

In [98]:
# création des interactions
formuleI = "1 + (" + "+".join(nomsvar) + ")**2"
PROB = pd.DataFrame({"Y":Y,"log":0.0,"BIC":0.0,"AIC":0.0,
                    "ridge":0.0,"lasso":0.0,"elast":0.0})


Xinter = dmatrix(formuleI, don, return_type="dataframe").iloc[:,1:].to_numpy()
Xinter.shape

(462, 45)

In [102]:
# Transformer Y en numpy si besoin
Y = don["Y"].to_numpy()
Y.shape

(462,)

In [103]:
cr = StandardScaler()
lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10, Cs=Cs_lasso,  solver="saga", max_iter=2000)
enetcv = LogisticRegressionCV(cv=10, penalty="elasticnet", l1_ratios = [0.5], n_jobs=10,  Cs=Cs_enet, solver="saga", max_iter=2000)
ridgecv = LogisticRegressionCV(cv=10, penalty="l2", n_jobs=10, Cs=Cs_ridge,  max_iter=1000)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])

nb=10
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=1234)

In [104]:
for app_index, val_index in skf.split(X,Y):
    Xapp = Xinter[app_index,:]
    Xtest = Xinter[val_index,:]
    Yapp = Y[app_index]
    ### log
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    #choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[], \
    #    direction="forward",crit="bic").fit(Xapp,Yapp)
    #PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    # choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[], \
    #    direction="forward",crit="aic").fit(Xapp,Yapp)
    #PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,\
                 Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,\
          l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2", \
            Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]

Comparaison des algos en prenant seuil s = 0,5

In [105]:
mc = pd.Series(0.0, index=PROB.columns[1:])
s = 0.5
for i in range(mc.shape[0]):
    mc.iloc[i] = sklm.zero_one_loss(PROB.Y, PROB.iloc[:,i+1]>s)

round(mc.sort_values(ascending=True),3)

ridge    0.260
elast    0.264
lasso    0.268
log      0.288
BIC      0.346
AIC      0.346
dtype: float64

Comparaison des algos en prenant seuil naturel

In [114]:
matsN = pd.DataFrame({"seuil": pd.Series(0.0, index=noms)})
noms = PROB.columns[1:]
nbr0 = PROB.Y.value_counts()[0]
for nom in noms:
    tmp = PROB.loc[:,nom].sort_values(ascending=True)
    s = (tmp.iloc[nbr0-1]+tmp.iloc[nbr0])/2
    matsN.loc[nom,"seuil"] = s
    matsN.loc[nom,"accuracy"] = sklm.accuracy_score(PROB.Y, PROB.loc[:,nom]>=s)
    confmat = sklm.confusion_matrix(PROB.Y, PROB.loc[:,nom]>=s)
  #  matsN.loc[nom, "tn"] = confmat[0,0]
  #  matsN.loc[nom, "tp"] = confmat[1,1]
  #  matsN.loc[nom, "fn"] = confmat[1,0]
  #  matsN.loc[nom, "fp"] = confmat[0,1]
    matsN.loc[nom,"sensitivity"] = confmat[1,1]/(confmat[1,1]+confmat[1,0])
    matsN.loc[nom,"specificity"] = confmat[0,0]/(confmat[0,0]+confmat[0,1])
    matsN.loc[nom,"medecin"] = matsN.loc[nom,"sensitivity"]+matsN.loc[nom,"specificity"]
    matsN.loc[nom,"F1"] = sklm.f1_score(PROB.Y, PROB.loc[:,nom]>=s)
print(matsN.round(3))

        seuil  accuracy  sensitivity  specificity  medecin     F1
log     0.410     0.680        0.538        0.755    1.292  0.538
BIC     0.000     0.346        1.000        0.000    1.000  0.514
AIC     0.000     0.346        1.000        0.000    1.000  0.514
ridge   0.426     0.723        0.600        0.788    1.388  0.600
lasso   0.430     0.719        0.594        0.785    1.379  0.594
elast   0.418     0.723        0.600        0.788    1.388  0.600
LassoL  0.000       NaN          NaN          NaN      NaN    NaN
RidgeL  0.000       NaN          NaN          NaN      NaN    NaN
EnetL   0.000       NaN          NaN          NaN      NaN    NaN
LassoA  0.000       NaN          NaN          NaN      NaN    NaN
RidgeA  0.000       NaN          NaN          NaN      NaN    NaN
EnetA   0.000       NaN          NaN          NaN      NaN    NaN


## Partie 3 — Feature engineering : termes polynomiaux (carrés et cubes)
On ajoute X², X³ pour les variables quantitatives, ce qui permet de modéliser des effets non-linéaires directs sur chaque variable (sans interactions croisées).

In [107]:
# création des termes carrés et cubiques
Xquanti = don.drop(columns="Y").\
                    select_dtypes(include=[np.number]).to_numpy()
Xcar = Xquanti**2
Xcub = Xquanti**3
formule = "~" + "+".join(nomsvar)
X = dmatrix(formule, don, return_type="dataframe").\
                                            iloc[:,1:].to_numpy()
Xpol = np.concatenate((X, Xcar, Xcub), axis=1)
Xpol.shape

(462, 25)

In [111]:
cr = StandardScaler()
lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10, Cs=Cs_lasso,  solver="saga", max_iter=2000)
enetcv = LogisticRegressionCV(cv=10, penalty="elasticnet", l1_ratios = [0.5], n_jobs=10,  Cs=Cs_enet, solver="saga", max_iter=20000)
ridgecv = LogisticRegressionCV(cv=10, penalty="l2", n_jobs=10, Cs=Cs_ridge,  max_iter=10000)
pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])

nb=10
skf = StratifiedKFold(n_splits=nb, shuffle=True, random_state=1234)

In [112]:
for app_index, val_index in skf.split(X,Y):
    Xapp = Xpol[app_index,:]
    Xtest = Xpol[val_index,:]
    Yapp = Y[app_index]
    ### log
    log = LogisticRegression(penalty=None,solver="newton-cholesky").fit(Xapp,Yapp)
    PROB.loc[val_index,"log"] = log.predict_proba(Xtest)[:,1]
    ### bic
    # choixbic = lss.LogisticRegressionSelectionFeatureIC(start=[], \
    #    direction="forward",crit="bic").fit(Xapp,Yapp)
    # PROB.loc[val_index, "BIC"] = choixbic.predict_proba(Xtest)[:,1]
    ### aic
    # choixaic = lss.LogisticRegressionSelectionFeatureIC(start=[], \
    #   direction="forward",crit="aic").fit(Xapp,Yapp)
    # PROB.loc[val_index, "AIC"] = choixaic.predict_proba(Xtest)[:,1]
    ### lasso
    cr = StandardScaler()
    Cs_lasso = grille(Xapp,Yapp, "lasso")
    lassocv =  LogisticRegressionCV(cv=10, penalty="l1", n_jobs=10,\
                 Cs=Cs_lasso,  solver="saga", max_iter=2000)
    pipe_lassocv = Pipeline(steps=[("cr", cr), ("lassocv", lassocv)])
    pipe_lassocv.fit(Xapp,Yapp)
    PROB.loc[val_index,"lasso"] = pipe_lassocv.predict_proba(Xtest)[:,1]
    ### elastic net
    cr = StandardScaler()
    Cs_enet = grille(Xapp,Yapp,"enet")
    enetcv=LogisticRegressionCV(cv=10,penalty="elasticnet",n_jobs=10,\
          l1_ratios=[0.5],Cs=Cs_enet,solver="saga",max_iter=2000)
    pipe_enetcv = Pipeline(steps=[("cr", cr), ("enetcv", enetcv)])
    pipe_enetcv.fit(Xapp,Yapp)
    PROB.loc[val_index,"elast"] = pipe_enetcv.predict_proba(Xtest)[:,1] 
    ### ridge
    cr = StandardScaler()
    Cs_ridge = grille(Xapp,Yapp,"ridge")
    ridgecv = LogisticRegressionCV(cv=10, penalty="l2", \
            Cs=Cs_ridge,  max_iter=1000)
    pipe_ridgecv = Pipeline(steps=[("cr", cr), ("ridgecv", ridgecv)])
    pipe_ridgecv.fit(Xapp,Yapp)
    PROB.loc[val_index,"ridge"] = pipe_ridgecv.predict_proba(Xtest)[:,1]

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:491: LinAlgWarning: The inner solver of NewtonCholeskySolver stumbled upon a singular or very ill-conditioned Hessian matrix at iteration #1. It will now resort to lbfgs instead.
Further options are to use another solver or to avoid such situation in the first place. Possible remedies are removing collinear features of X or increasing the penalization strengths.
The original Linear Algebra message was:
Ill-conditioned matrix (rcond=1.87382e-18): result may not be accurate.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:195: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res)
C:\ProgramData\anaconda3\L

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_sag.py:350: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\linear_model\_glm\_newton_solver.py:491: LinAlgWarning: The

In [113]:
mc = pd.Series(0.0, index=PROB.columns[1:])
s = 0.5
for i in range(mc.shape[0]):
    mc.iloc[i] = sklm.zero_one_loss(PROB.Y, PROB.iloc[:,i+1]>s)

round(mc.sort_values(ascending=True),3)

lasso    0.266
ridge    0.271
elast    0.277
log      0.288
BIC      0.346
AIC      0.346
dtype: float64